In [ ]:
#| default_exp encoder

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, numpy as np
from pathlib import Path
from fastcore.all import ifnone, patch
from laghu.core import log, resolve, model_files, st_map
from laghu.unigram import tokenizer, Unigram

## LeanModel

`LeanModel` has the parts of `StaticModel` that read a model: `tokenize`, `encode`, `encode_as_sequence`, and the attributes `embedding`,
`weights`, `token_mapping`, `normalize`, `config`, `dim`, `median_token_length`, `unk_token_id`, `tokens`. Encoding uses model2vec's numpy operations and batch boundaries. This preserves output bits for float16 tables and the float64 row of an empty text. Storage differs:

- `embedding` (and `weights` and `mapping`, when a vocabulary-quantized model has them) is an `np.memmap` of `model.safetensors`, read-only;
- the tokenizer is `Unigram` for a Unigram model, `tokenizers` for anything else;
- `tokens` is built on first access (500k strings are about 40 MB).

`encode([])` returns an empty `(0, dim)` array, where model2vec raises. `LeanModel` accepts `use_multiprocessing` and ignores it.

In [ ]:
#| export
class LeanModel:
    "model2vec's `StaticModel` for reading a model, with its table mapped from the file and, for Unigram, no trie."
    def __init__(self, folder, normalize=None, lean=True):
        f = model_files(folder); self.folder = Path(folder)
        self.config = json.loads(f.cfg.read_text())
        self.embedding, self.weights, self.token_mapping = st_map(f.emb, [f.name, 'weights', 'mapping'])
        if self.embedding is None: raise KeyError(f'{f.emb}: no tensor {f.name!r}')
        self.tokenizer = tokenizer(f.tok, lean)
        if self.token_mapping is None and self.tokenizer.n != len(self.embedding):
            raise ValueError(f'Number of tokens ({self.tokenizer.n}) does not match number of vectors ({len(self.embedding)}).')
        self.median_token_length, self.unk_token_id = self.tokenizer.median, self.tokenizer.unk_token_id
        self.normalize = ifnone(normalize, self.config.get('normalize', False))
        self.config['normalize'] = self.normalize

    @classmethod
    def from_pretrained(cls, path, token=None, normalize=None, subfolder=None, force_download=False, lean=True):
        "A `LeanModel` for a folder or Hugging Face repo id `path`; a cached repo loads without the network."
        return cls(resolve(path, token=token, subfolder=subfolder, force_download=force_download), normalize=normalize, lean=lean)

    @property
    def dim(self): return self.embedding.shape[1]
    @property
    def embedding_dtype(self): return np.dtype(self.embedding.dtype).name
    @property
    def lean(self): return isinstance(self.tokenizer, Unigram)
    @property
    def tokens(self):
        "Every token, in id order, as `StaticModel.tokens`."
        if not hasattr(self, '_tokens'): v = self.tokenizer.vocab(); self._tokens = tuple(sorted(v, key=v.get))
        return self._tokens

    def __repr__(self):
        return f"{type(self).__name__}({str(self.folder)!r}, {len(self.embedding)}×{self.dim} {self.embedding_dtype}, {'unigram' if self.lean else 'tokenizers'})"

`tokenize` follows model2vec's order. It truncates each text to `max_length × median_token_length` characters and tokenizes it.
It then drops the unknown id where the model names one (WordPiece and WordLevel do, Unigram does not) and keeps the first `max_length` ids.

In [ ]:
#| export
def _batches(xs, bs): return (xs[i:i + bs] for i in range(0, len(xs), bs))

@patch
def tokenize(self:LeanModel, sentences, max_length=None):
    "The ids of each of `sentences`, as `StaticModel.tokenize`."
    if max_length is not None: m = max_length * self.median_token_length; sentences = [s[:m] for s in sentences]
    ids = self.tokenizer(sentences)
    if self.unk_token_id is not None: ids = [[k for k in o if k != self.unk_token_id] for o in ids]
    return [o[:max_length] for o in ids] if max_length is not None else ids

@patch
def _rows(self:LeanModel, ids):
    "The vectors of `ids`, remapped and weighted when the model is vocabulary-quantized."
    emb = self.embedding[ids if self.token_mapping is None else self.token_mapping[ids]]
    return emb * self.weights[ids][:, None] if self.weights is not None else emb

@patch
def _encode_batch(self:LeanModel, sentences, max_length):
    "One batch: model2vec stacks the means, so an empty text's float64 zero row makes its whole batch float64."
    out = np.stack([self._rows(o).mean(axis=0) if o else np.zeros(self.dim) for o in self.tokenize(sentences, max_length)])
    return out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-32) if self.normalize else out

@patch
def encode(self:LeanModel, sentences, show_progress_bar=False, max_length=512, batch_size=1024, use_multiprocessing=True, multiprocessing_threshold=10_000, **kwargs):
    "Each sentence's mean token vector (unit length when `normalize`), as `StaticModel.encode`; one string gives one vector."
    one = isinstance(sentences, str)
    xs = [sentences] if one else list(sentences)
    if not xs: return np.zeros((0, self.dim), self.embedding.dtype)
    out = np.concatenate([self._encode_batch(b, max_length) for b in _batches(xs, batch_size)], axis=0)
    return out[0] if one else out

@patch
def encode_as_sequence(self:LeanModel, sentences, max_length=None, batch_size=1024, show_progress_bar=False, use_multiprocessing=True, multiprocessing_threshold=10_000):
    "Each sentence's token vectors, `(n_tokens, dim)`, as `StaticModel.encode_as_sequence`."
    one = isinstance(sentences, str)
    xs = [sentences] if one else list(sentences)
    out = [self._rows(o) if o else np.zeros((0, self.dim)) for b in _batches(xs, batch_size) for o in self.tokenize(b, max_length)]
    return out[0] if one else out

## load

`load` returns a `LeanModel`. If loading fails, it logs the cause and tries model2vec's `StaticModel`. If that also fails, it raises the original laghu error. `fallback=False` disables the fallback. Only the fallback imports model2vec, and it requires `laghu[fallback]`.

In [ ]:
#| export
def load(path, normalize=None, token=None, subfolder=None, force_download=False, lean=True, fallback=True):
    "A `LeanModel` for `path`, else model2vec's `StaticModel` with a logged warning; `fallback=False` raises instead."
    try: return LeanModel.from_pretrained(path, token=token, normalize=normalize, subfolder=subfolder, force_download=force_download, lean=lean)
    except Exception as e:
        if not fallback: raise
        log.warning(f'laghu: cannot read {path} ({e!r}); loading it with model2vec instead')
        try:
            from model2vec import StaticModel
            return StaticModel.from_pretrained(path, token=token, normalize=normalize, subfolder=subfolder, force_download=force_download)
        except Exception as e2: raise e from e2

## Tests

`same_as_model2vec` loads a model both ways. It compares `tokenize` and `encode` on texts, float32 and float16, bit for bit, and
`encode_as_sequence` on a sample. laghu exports it for checking a model of your own.

In [ ]:
#| export
def same_as_model2vec(path, txts=None, max_length=512, seq=200):
    "Whether `load(path)` matches model2vec on `txts` (default `corpus()`): a dict of the checks."
    from model2vec import StaticModel
    from laghu.core import corpus
    txts = ifnone(txts, corpus())
    lm, sm = load(path, fallback=False), StaticModel.from_pretrained(str(resolve(path)), force_download=False)
    a, b = lm.encode(txts, max_length=max_length), sm.encode(txts, max_length=max_length)
    sa, sb = lm.encode_as_sequence(txts[:seq]), sm.encode_as_sequence(txts[:seq])
    return dict(lean=lm.lean, n=len(txts), ids=lm.tokenize(txts, max_length) == sm.tokenize(txts, max_length), dtype=a.dtype == b.dtype,
        f32=np.array_equal(a.astype(np.float32), b.astype(np.float32)), f16=np.array_equal(a.astype(np.float16), b.astype(np.float16)),
        exact=a.dtype == b.dtype and np.array_equal(a, b), seq=all(x.dtype == y.dtype and np.array_equal(x, y) for x, y in zip(sa, sb)),
        median=lm.median_token_length == sm.median_token_length, unk=lm.unk_token_id == sm.unk_token_id, tokens=lm.tokens == tuple(sm.tokens))

In [ ]:
import importlib.util
from laghu.core import try_resolve, corpus, ODD
M2V = importlib.util.find_spec('model2vec') is not None   # the `fallback` extra; the tests that compare with it skip without it
txts = corpus(1500)
ok = lambda r: all(v for k, v in r.items() if k not in ('lean', 'n'))

In [ ]:
if M2V and try_resolve('minishlab/potion-multilingual-128M'):
    m = load('minishlab/potion-multilingual-128M')
    assert isinstance(m, LeanModel) and m.lean and isinstance(m.embedding, np.memmap) and not m.embedding.flags.writeable
    r = same_as_model2vec('minishlab/potion-multilingual-128M', txts); print(r); assert ok(r) and r['lean']
    for ml in (None, 8): assert m.tokenize(ODD, ml) == __import__('model2vec').StaticModel.from_pretrained(str(m.folder), force_download=False).tokenize(ODD, ml)

In [ ]:
if M2V and try_resolve('minishlab/potion-base-8M'):   # WordPiece: tokenizers tokenizes, the table is still mapped
    r = same_as_model2vec('minishlab/potion-base-8M', txts); print(r); assert ok(r) and not r['lean']

In [ ]:
if M2V and try_resolve('minishlab/potion-code-16M-v2'):   # a float16 table
    m = load('minishlab/potion-code-16M-v2'); assert m.embedding_dtype == 'float16'
    r = same_as_model2vec('minishlab/potion-code-16M-v2', txts); print(r); assert ok(r)

The odd shapes: a single string, an empty list, `normalize` overridden, an empty text's float64 row, and the fallback.

In [ ]:
if try_resolve('minishlab/potion-base-8M'):
    m = load('minishlab/potion-base-8M')
    assert m.encode('hello').shape == (m.dim,) and m.encode([]).shape == (0, m.dim)
    assert m.encode(['', 'hello']).dtype == np.float64 and m.encode(['hello']).dtype == np.float32   # as model2vec: the zero row is float64
    assert not np.allclose(np.linalg.norm(load('minishlab/potion-base-8M', normalize=False).encode(['hello world'])), 1)
    assert m.encode_as_sequence('') .shape == (0, m.dim) and m.encode_as_sequence('hello world').shape[1] == m.dim

In [ ]:
import logging, tempfile, shutil
from laghu.core import st_header
from fastcore.all import Path
if f := try_resolve('minishlab/potion-base-8M'):
    bf = Path(tempfile.mkdtemp())
    for o in ('config.json', 'tokenizer.json'): shutil.copy(f/o, bf/o)
    hdr, at = st_header(f/'model.safetensors'); hdr['embeddings']['dtype'] = 'BF16'; hdr['embeddings']['shape'][1] *= 2
    h = json.dumps(hdr).encode(); h += b' ' * (-len(h) % 8)
    (bf/'model.safetensors').write_bytes(len(h).to_bytes(8, 'little') + h + (f/'model.safetensors').read_bytes()[at:])
    try: load(bf); raise AssertionError('loaded a BF16 table')
    except Exception as e: assert 'BF16' in str(e), e        # model2vec fails too, so laghu's error is the one raised
    shutil.copy(f/'model.safetensors', bf/'model.safetensors')
    (bf/'config.json').write_text('{not json')            # laghu cannot read the config; model2vec cannot either
    try: load(bf); raise AssertionError('loaded a broken config')
    except json.JSONDecodeError: pass

In [ ]:
if M2V and try_resolve('minishlab/potion-base-8M'):
    from unittest import mock
    from model2vec import StaticModel
    with mock.patch.object(LeanModel, '__init__', side_effect=RuntimeError('boom')):
        logging.getLogger('laghu').propagate = True
        m = load('minishlab/potion-base-8M')
    assert isinstance(m, StaticModel)

Two models built here from parts, to reach the paths no public potion model takes: a vocabulary-quantized model (a `mapping` from token
to one of 1,000 rows, and per-token `weights`), and a Unigram model on t5-small's tokenizer (no model type, WhitespaceSplit before
Metaspace, 103 added tokens) with a random float16 table.

In [ ]:
from laghu.core import st_write
rng = np.random.default_rng(0)
if M2V and (f := try_resolve('minishlab/potion-base-8M')):
    vq = Path(tempfile.mkdtemp()); shutil.copy(f/'tokenizer.json', vq/'tokenizer.json')
    emb, = st_map(f/'model.safetensors', ['embeddings'])
    st_write(vq/'model.safetensors', dict(embeddings=np.array(emb[:1000]), mapping=rng.integers(0, 1000, len(emb)).astype(np.int64),
                                          weights=rng.random(len(emb)).astype(np.float32)))
    (vq/'config.json').write_text(json.dumps(dict(normalize=True, vocabulary_quantization=1000)))
    m = load(vq); assert m.token_mapping is not None and m.weights is not None
    r = same_as_model2vec(vq, txts); print(r); assert ok(r)

In [ ]:
def hub_file(repo, fn):
    from huggingface_hub import hf_hub_download
    try: return Path(hf_hub_download(repo, fn))
    except Exception as e: print(f'skipping {repo}: {e!r}')

if M2V and (fn := hub_file('google-t5/t5-small', 'tokenizer.json')):
    t5 = Path(tempfile.mkdtemp()); shutil.copy(fn, t5/'tokenizer.json')
    st_write(t5/'model.safetensors', dict(embeddings=rng.standard_normal((32100, 64)).astype(np.float16)))
    (t5/'config.json').write_text(json.dumps(dict(normalize=False)))
    m = load(t5); assert m.lean and m.embedding_dtype == 'float16'
    r = same_as_model2vec(t5, txts); print(r); assert ok(r)